# ICL Attention Visualization

Visualize how the transformer attends to context when making predictions.

## Setup

In [ ]:
import torch

print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import sys
import os

PROJECT_PATH = '/content/drive/MyDrive/micro-research/ICL'
if PROJECT_PATH not in sys.path:
    sys.path.insert(0, PROJECT_PATH)
os.chdir(PROJECT_PATH)

print(f"Working directory: {os.getcwd()}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from data import make_batch_xy, make_batch_xy_fixed_padded
from model import create_model
from attention import (
    get_attention_weights,
    plot_attention_heatmap,
    plot_query_attention,
    plot_all_layers_query_attention,
    plot_attention_grid,
    analyze_attention_patterns,
    print_attention_analysis,
)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print("Imports successful!")

## Load Trained Model

Either load a checkpoint or train a new model.

In [ ]:
# Configuration (must match the trained model)
CONFIG = {
    'd': 5,
    'n_ctx_max': 128,
    'd_model': 256,
    'n_heads': 8,
    'n_layers': 6,
}

In [ ]:
# Option 1: Load from checkpoint
checkpoint_path = os.path.join(PROJECT_PATH, 'model_checkpoint.pt')

if os.path.exists(checkpoint_path):
    checkpoint = torch.load(checkpoint_path, map_location=device)
    if 'config' in checkpoint:
        CONFIG = checkpoint['config']
    
    model = create_model(
        d=CONFIG['d'],
        n_ctx_max=CONFIG['n_ctx_max'],
        d_model=CONFIG['d_model'],
        n_heads=CONFIG['n_heads'],
        n_layers=CONFIG['n_layers'],
        device=device,
    )
    
    state_dict = checkpoint.get('model_state_dict', checkpoint)
    model.load_state_dict(state_dict)
    print(f"Loaded checkpoint from {checkpoint_path}")
else:
    print(f"No checkpoint found at {checkpoint_path}")
    print("Run ICL_colab.ipynb first to train a model, or train one below.")

In [ ]:
# Option 2: Train a quick model (if no checkpoint)
# Uncomment to train a smaller/faster model for visualization

# from train import train
# model = train(
#     d=5,
#     n_ctx_max=64,
#     d_model=128,
#     n_heads=4,
#     n_layers=4,
#     steps=5000,
#     batch_size=256,
# )
# CONFIG = {'d': 5, 'n_ctx_max': 64, 'd_model': 128, 'n_heads': 4, 'n_layers': 4}

## Generate Test Data

In [ ]:
# Generate a single example for visualization
n_ctx = 32  # Use a moderate context length for clearer visualization

seq, yq, pad_mask = make_batch_xy_fixed_padded(
    batch_size=1,
    n_ctx=n_ctx,
    d=CONFIG['d'],
    n_ctx_max=CONFIG['n_ctx_max'],
    noise_std=0.05,
)
seq = seq.to(device)
pad_mask = pad_mask.to(device)

print(f"Sequence shape: {seq.shape}")
print(f"Context length: {n_ctx}")
print(f"True y_query: {yq.item():.4f}")

## Extract Attention Weights

In [ ]:
# Get attention weights from all layers
with torch.no_grad():
    pred, attentions = get_attention_weights(model, seq, pad_mask=pad_mask)

print(f"Predicted y_query: {pred.item():.4f}")
print(f"True y_query: {yq.item():.4f}")
print(f"Error: {abs(pred.item() - yq.item()):.4f}")
print(f"\nCaptured {len(attentions)} layers of attention")
print(f"Attention shape per layer: {attentions[0].shape}")

## Attention Analysis

In [ ]:
# Compute attention statistics
analysis = analyze_attention_patterns(attentions, n_ctx=n_ctx)
print_attention_analysis(analysis, n_ctx=n_ctx)

## Visualizations

### Full Attention Grid (All Layers & Heads)

In [ ]:
# Plot attention heatmaps for all layers and heads
# Red lines mark the query token position
fig = plot_attention_grid(attentions, n_ctx=n_ctx)
plt.suptitle(f"Attention Patterns (n_ctx={n_ctx})", y=1.02, fontsize=14)
plt.show()

### Query Token Attention (What does the query attend to?)

In [ ]:
# Plot what the query token attends to across layers
fig = plot_all_layers_query_attention(attentions, n_ctx=n_ctx)
plt.suptitle(f"Query Attention to Context (n_ctx={n_ctx})", y=1.02, fontsize=14)
plt.show()

### Single Layer/Head Detail

In [ ]:
# Detailed view of a specific layer and head
layer_to_view = CONFIG['n_layers'] - 1  # Last layer
head_to_view = 0

fig, ax = plt.subplots(figsize=(10, 8))
plot_attention_heatmap(
    attentions[layer_to_view],
    layer=layer_to_view,
    head=head_to_view,
    n_ctx=n_ctx,
    ax=ax,
)
plt.show()

## Compare Attention at Different Context Lengths

In [ ]:
# See how attention changes with more context
context_lengths = [4, 16, 32, 64]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for idx, n in enumerate(context_lengths):
    ax = axes[idx // 2, idx % 2]
    
    # Generate data with this context length
    seq_n, yq_n, pad_mask_n = make_batch_xy_fixed_padded(
        batch_size=1,
        n_ctx=n,
        d=CONFIG['d'],
        n_ctx_max=CONFIG['n_ctx_max'],
    )
    seq_n = seq_n.to(device)
    pad_mask_n = pad_mask_n.to(device)
    
    with torch.no_grad():
        _, attn_n = get_attention_weights(model, seq_n, pad_mask=pad_mask_n)
    
    # Plot last layer, average over heads
    last_layer_attn = attn_n[-1][0].mean(dim=0).cpu().numpy()  # Average over heads
    
    im = ax.imshow(last_layer_attn, cmap='Blues', aspect='auto', vmin=0)
    ax.set_title(f'n_ctx={n} (Last Layer, Avg Heads)')
    ax.set_xlabel('Key Position')
    ax.set_ylabel('Query Position')
    ax.axhline(y=n, color='red', linestyle='--', alpha=0.5)
    ax.axvline(x=n, color='red', linestyle='--', alpha=0.5)
    plt.colorbar(im, ax=ax, fraction=0.046)

plt.tight_layout()
plt.suptitle('Attention Patterns at Different Context Lengths', y=1.02, fontsize=14)
plt.show()

## Query Attention vs Context Position

In [ ]:
# Does the model attend uniformly or prefer certain positions?
n_ctx = 64

seq, yq, pad_mask = make_batch_xy_fixed_padded(
    batch_size=16,  # Average over multiple examples
    n_ctx=n_ctx,
    d=CONFIG['d'],
    n_ctx_max=CONFIG['n_ctx_max'],
)
seq = seq.to(device)
pad_mask = pad_mask.to(device)

with torch.no_grad():
    _, attentions = get_attention_weights(model, seq, pad_mask=pad_mask)

# Average query attention across batch and heads
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

for layer in range(min(6, len(attentions))):
    ax = axes[layer // 3, layer % 3]
    
    # Query attention to context positions
    query_attn = attentions[layer][:, :, n_ctx, :n_ctx]  # (B, n_heads, n_ctx)
    avg_attn = query_attn.mean(dim=(0, 1)).cpu().numpy()  # Average over batch and heads
    
    ax.bar(range(n_ctx), avg_attn, alpha=0.7)
    ax.axhline(y=1/n_ctx, color='red', linestyle='--', label='Uniform')
    ax.set_xlabel('Context Position')
    ax.set_ylabel('Avg Attention')
    ax.set_title(f'Layer {layer}')
    ax.legend()

plt.tight_layout()
plt.suptitle(f'Query Attention Distribution (n_ctx={n_ctx}, averaged)', y=1.02)
plt.show()

## Head Specialization Analysis

In [ ]:
# Check if different heads have different roles
n_ctx = 32
n_samples = 64

seq, yq, pad_mask = make_batch_xy_fixed_padded(
    batch_size=n_samples,
    n_ctx=n_ctx,
    d=CONFIG['d'],
    n_ctx_max=CONFIG['n_ctx_max'],
)
seq = seq.to(device)
pad_mask = pad_mask.to(device)

with torch.no_grad():
    _, attentions = get_attention_weights(model, seq, pad_mask=pad_mask)

# For each head in the last layer, compute entropy of query attention
last_attn = attentions[-1]  # (B, n_heads, L, L)
query_attn = last_attn[:, :, n_ctx, :n_ctx]  # (B, n_heads, n_ctx)

# Entropy: -sum(p * log(p))
eps = 1e-10
entropy = -(query_attn * (query_attn + eps).log()).sum(dim=-1)  # (B, n_heads)
max_entropy = np.log(n_ctx)

fig, ax = plt.subplots(figsize=(10, 5))
entropy_mean = entropy.mean(dim=0).cpu().numpy()
entropy_std = entropy.std(dim=0).cpu().numpy()

x = range(CONFIG['n_heads'])
ax.bar(x, entropy_mean, yerr=entropy_std, capsize=5, alpha=0.7)
ax.axhline(y=max_entropy, color='red', linestyle='--', label=f'Max entropy (uniform) = {max_entropy:.2f}')
ax.set_xlabel('Head')
ax.set_ylabel('Attention Entropy')
ax.set_title(f'Last Layer: Head Attention Entropy (n_ctx={n_ctx})')
ax.legend()
ax.set_xticks(x)
plt.show()

print("\nLow entropy = concentrated attention (specific positions)")
print("High entropy = diffuse attention (uniform over context)")